In [19]:
import numpy as np
import scipy
import matplotlib.pyplot as plt
from qiskit.quantum_info import Pauli

np.set_printoptions(linewidth=200, precision=4, suppress=True)

In [20]:
# Initialize Pauli matrices

pauli_x = Pauli('X')
pauli_y = Pauli('Y')
pauli_z = Pauli('Z')
identity = Pauli('I')

X = pauli_x.to_matrix()
Y = pauli_y.to_matrix()
Z = pauli_z.to_matrix()
I = identity.to_matrix()

Ix1 = np.kron(X/2,I)
Iy1 = np.kron(Y/2,I)
Iz1 = np.kron(Z/2,I)
Ix2 = np.kron(I,X/2)
Iy2 = np.kron(I,Y/2)
Iz2 = np.kron(I,Z/2)

J = 696

Unitary matrix --> Hamiltonian and pulse parameters = difficult

Pulse parameters --> Unitary matrix = easy

Method: change pulse parameters, calculate how similar the resulting Unitary matrix is to Target unitary matrix, compare and optimize

In [21]:
# Parameters for a 2 qubit Rx(pi/2) pulse, Hydrogen and Phosphorus

R1=R2=1
w_rf=6250
phi_rf=np.pi/2
t_rf=40e-6

w0=w_rf
w1_H=w1_P=6250


In [22]:
# 2 qubit Unitary matrix calculation from system Hamiltonian

def twoqubit_unitary(R1,R2,w_rf,phi_rf,t_rf,w0,w1_H,w1_P):
    H0 = 2*np.pi*(w0-w_rf)*Iz1+2*np.pi*(w0-w_rf)*Iz2

    HJ = 2*np.pi*J*(np.kron(Z,Z)/4)

    H1 = 2*np.pi*R1*w1_H*(np.cos(phi_rf)*Ix1+np.sin(phi_rf)*Iy1)+2*np.pi*R2*w1_P*(np.cos(phi_rf)*Ix2+np.sin(phi_rf)*Iy2)

    return scipy.linalg.expm(-1j*(H0+HJ+H1)*t_rf)



In [23]:
Rypi2 = twoqubit_unitary(R1,R2,w_rf,phi_rf,t_rf,w0,w1_H,w1_P)

In [24]:
rho = Iz1

rho_f = Rypi2 @ rho @ Rypi2.conj().T

rho_f

array([[ 0.0003+0.j    ,  0.    -0.j    ,  0.4993-0.0219j, -0.    -0.0139j],
       [ 0.    +0.j    ,  0.0003+0.j    , -0.    -0.0139j,  0.4993+0.0219j],
       [ 0.4993+0.0219j, -0.    +0.0139j, -0.0003+0.j    ,  0.    +0.j    ],
       [-0.    +0.0139j,  0.4993-0.0219j,  0.    -0.j    , -0.0003+0.j    ]])

In [25]:
print(np.trace(rho_f @ Ix1))
print(np.trace(rho_f @ Iy1))
print(np.trace(rho_f @ Iz1))


(0.9986568311717521+0j)
(-6.591949208711867e-17+0j)
(0.000608575287973024+0j)


1. Create unitary operator

$$
U = R_{-z}^2\left(\frac{\pi}{4}\right)
    R_{-y}^2\left(\frac{\pi}{4}\right)
    R_{z}^1\left(\frac{\pi}{4}\right)
    R_{y}^1\left(\frac{\pi}{4}\right)
$$

In [26]:
# Rotation Rz through euler angles Rz(theta)=Rx(pi/2)Ry(theta)Rx(-pi/2)

R1 = twoqubit_unitary(0,1,6250,0,40e-6,6250,6250,6250)
R2 = twoqubit_unitary(0,1,6250,-np.pi/2,20e-6,6250,6250,6250)
R3 = twoqubit_unitary(0,1,6250,np.pi,40e-6,6250,6250,6250)

R2_zpi4 = R1 @ R2 @ R3

print(R2_zpi4)



[[ 0.9409+0.3308j  0.0729-0.j      0.    +0.j      0.    +0.j    ]
 [-0.0729-0.j      0.9409-0.3308j  0.    +0.j      0.    +0.j    ]
 [ 0.    +0.j      0.    +0.j      0.8983+0.4334j -0.0724-0.j    ]
 [ 0.    +0.j      0.    +0.j      0.0724+0.j      0.8983-0.4334j]]


In [27]:
# Rotation Ry

R2_ypi4 = twoqubit_unitary(0,1,6250,-np.pi/2,20e-6,6250,6250,6250)

In [28]:
# Rotation Rz through euler angles Rz(theta)=Rx(pi/2)Ry(theta)Rx(-pi/2)

R1 = twoqubit_unitary(1,0,6250,0,40e-6,6250,6250,6250)
R2 = twoqubit_unitary(1,0,6250,np.pi/2,20e-6,6250,6250,6250)
R3 = twoqubit_unitary(1,0,6250,np.pi,40e-6,6250,6250,6250)

R1zpi4 = R1 @ R2 @ R3

print(R1zpi4)


[[ 0.8983-0.4334j  0.    +0.j      0.0724-0.j      0.    +0.j    ]
 [ 0.    +0.j      0.9409-0.3308j  0.    +0.j     -0.0729-0.j    ]
 [-0.0724+0.j      0.    +0.j      0.8983+0.4334j  0.    +0.j    ]
 [ 0.    +0.j      0.0729-0.j      0.    +0.j      0.9409+0.3308j]]


In [29]:
# Rotation Ry

R1ypi4 = twoqubit_unitary(1,0,6250,np.pi/2,20e-6,6250,6250,6250)

U achieved using hard pulses

In [30]:
U = R2_zpi4 @ R2_ypi4 @ R1zpi4 @ R1ypi4

print(U)

[[ 0.8392-0.1136j  0.3991-0.0133j -0.2826+0.0629j -0.1991-0.j    ]
 [-0.3088+0.2879j  0.6379-0.4843j  0.0969-0.107j  -0.3259+0.2307j]
 [ 0.1646+0.2581j  0.0969+0.107j   0.5435+0.7115j  0.1871+0.221j ]
 [-0.1047-0.j      0.4162-0.0706j -0.305 +0.0258j  0.8392-0.1136j]]


Strongly Modulated Pulse optimization

In [31]:
U_target = U

In [32]:
# SMP following SpinQ manual 

# constants
n = 2
w0 = 6250
w1_max = 6250
J = 696
T = 200e-6
N = 50  # maybe add more divisions?
dt = T/N

# initialize
R = np.full(N, 1.0, dtype=float)
phi = np.zeros(N, dtype=float)
omega_rf = np.full(N, w0, dtype=float)
dts = np.full(N, dt, dtype=float)

# Compute fidelity aka loss function

def fidelity(R,phi,omega_rf,dts):
    U = np.eye(2**n)
    for k in range(N):
        H_0 = 2*np.pi*(w0-omega_rf[k])*Iz1+2*np.pi*(w0-omega_rf[k])*Iz2
        H_j = 2*np.pi*J*(np.kron(Z,Z)/4)
        H_rf = 2*np.pi*R[k]*w1_max*(np.cos(phi[k])*Ix1+np.sin(phi[k])*Iy1)+2*np.pi*R[k]*w1_max*(np.cos(phi[k])*Ix2+np.sin(phi[k])*Iy2)
        H_total = H_0 + H_j + H_rf
        U = scipy.linalg.expm(-1j*H_total*dts[k]) @ U
    return np.abs(np.trace(U_target.conj().T @ U)) / (2**n)

# Gradient descent algorithm

Q = 1-fidelity(R,phi,omega_rf,dts)

# Calculate numerical gradient

def num_grad(R, phi, omega_rf, dts, d=1e-2):
    grad_R = np.zeros_like(R)
    grad_phi = np.zeros_like(phi)
    grad_omega = np.zeros_like(omega_rf)
    #grad_dt = np.zeros_like(dts)

    eps = 1e-6
    # per-component increments
    dR = np.maximum(d*R, eps)
    dphi = np.maximum(d*np.abs(phi), eps)
    dw = np.maximum(d*omega_rf, eps)
    #ddt = np.maximum(d*dts, eps)

    Q0 = 1 - fidelity(R, phi, omega_rf, dts)

    for k in range(N):
        R_d = R.copy()
        R_d[k] += dR[k]
        grad_R[k] = (1-fidelity(R_d, phi, omega_rf, dts) - Q0)/dR[k]

        phi_d = phi.copy()
        phi_d[k] += dphi[k]
        grad_phi[k] = (1-fidelity(R, phi_d, omega_rf, dts) - Q0)/dphi[k]

        omega_d = omega_rf.copy()
        omega_d[k] += dw[k]
        grad_omega[k] = (1-fidelity(R, phi, omega_d, dts) - Q0)/dw[k]

        #dt_d = dts.copy()
        #dt_d[k] += ddt[k]
        #grad_dt[k] = (1-fidelity(R, phi, omega_rf, dt_d) - Q0)/ddt[k]

    return grad_R, grad_phi, grad_omega#, grad_dt


# Gradient descent loop

learn_rate = 0.05 # 500 iterations 0.01 is small - 38 minutes - fidelity = 0.7244; 500 iterations 0.05 - plateau at fidelity = 0.74
# consider subdividing pulse more: +N
iter = 500

for i in range(iter):
    grad_R,grad_phi,grad_omega = num_grad(R,phi,omega_rf,dts) #,grad_dt
    
    R -= learn_rate * grad_R
    phi -= learn_rate * grad_phi
    omega_rf -= learn_rate * grad_omega
    #dts -= learn_rate * grad_dt
    
    R = np.clip(R,0,1)
    phi = phi % (2*np.pi)
    
    if i % 10 == 0 or i == iter-1:
        print(f"Iteration {i}, Fidelity={fidelity(R,phi,omega_rf,dts):.6f}")




Iteration 0, Fidelity=0.258031
Iteration 10, Fidelity=0.335646
Iteration 20, Fidelity=0.417444
Iteration 30, Fidelity=0.495428
Iteration 40, Fidelity=0.562964
Iteration 50, Fidelity=0.616736
Iteration 60, Fidelity=0.656741
Iteration 70, Fidelity=0.685020
Iteration 80, Fidelity=0.704293
Iteration 90, Fidelity=0.717103
Iteration 100, Fidelity=0.725476
Iteration 110, Fidelity=0.730889
Iteration 120, Fidelity=0.734364
Iteration 130, Fidelity=0.736584
Iteration 140, Fidelity=0.738000
Iteration 150, Fidelity=0.738900
Iteration 160, Fidelity=0.739473
Iteration 170, Fidelity=0.739837
Iteration 180, Fidelity=0.740070
Iteration 190, Fidelity=0.740218
Iteration 200, Fidelity=0.740313
Iteration 210, Fidelity=0.740375
Iteration 220, Fidelity=0.740416
Iteration 230, Fidelity=0.740443
Iteration 240, Fidelity=0.740461
Iteration 250, Fidelity=0.740474
Iteration 260, Fidelity=0.740484
Iteration 270, Fidelity=0.740492
Iteration 280, Fidelity=0.740498
Iteration 290, Fidelity=0.740504
Iteration 300, Fideli

In [33]:
U_cnot = np.array([[1,0,0,0],
                   [0,1,0,0],
                   [0,0,0,1],
                   [0,0,1,0]])

In [34]:
U_target = U_cnot

# constants
n = 2
w0 = 6250
w1_max = 6250
J = 696
T = 200e-6
N = 50  # maybe add more divisions?
dt = T/N

# initialize
R = np.full(N, 1.0, dtype=float)
phi = np.zeros(N, dtype=float)
omega_rf = np.full(N, w0, dtype=float)
dts = np.full(N, dt, dtype=float)

# Compute fidelity aka loss function

def fidelity(R,phi,omega_rf,dts):
    U = np.eye(2**n)
    for k in range(N):
        H_0 = 2*np.pi*(w0-omega_rf[k])*Iz1+2*np.pi*(w0-omega_rf[k])*Iz2
        H_j = 2*np.pi*J*(np.kron(Z,Z)/4)
        H_rf = 2*np.pi*R[k]*w1_max*(np.cos(phi[k])*Ix1+np.sin(phi[k])*Iy1)+2*np.pi*R[k]*w1_max*(np.cos(phi[k])*Ix2+np.sin(phi[k])*Iy2)
        H_total = H_0 + H_j + H_rf
        U = scipy.linalg.expm(-1j*H_total*dts[k]) @ U
    return np.abs(np.trace(U_target.conj().T @ U)) / (2**n)

# Gradient descent algorithm

Q = 1-fidelity(R,phi,omega_rf,dts)

# Calculate numerical gradient

def num_grad(R, phi, omega_rf, dts, d=1e-2):
    grad_R = np.zeros_like(R)
    grad_phi = np.zeros_like(phi)
    grad_omega = np.zeros_like(omega_rf)
    #grad_dt = np.zeros_like(dts)

    eps = 1e-6
    # per-component increments
    dR = np.maximum(d*R, eps)
    dphi = np.maximum(d*np.abs(phi), eps)
    dw = np.maximum(d*omega_rf, eps)
    #ddt = np.maximum(d*dts, eps)

    Q0 = 1 - fidelity(R, phi, omega_rf, dts)

    for k in range(N):
        R_d = R.copy()
        R_d[k] += dR[k]
        grad_R[k] = (1-fidelity(R_d, phi, omega_rf, dts) - Q0)/dR[k]

        phi_d = phi.copy()
        phi_d[k] += dphi[k]
        grad_phi[k] = (1-fidelity(R, phi_d, omega_rf, dts) - Q0)/dphi[k]

        omega_d = omega_rf.copy()
        omega_d[k] += dw[k]
        grad_omega[k] = (1-fidelity(R, phi, omega_d, dts) - Q0)/dw[k]

        #dt_d = dts.copy()
        #dt_d[k] += ddt[k]
        #grad_dt[k] = (1-fidelity(R, phi, omega_rf, dt_d) - Q0)/ddt[k]

    return grad_R, grad_phi, grad_omega#, grad_dt


# Gradient descent loop

learn_rate = 0.05 # 500 iterations 0.01 is small - 38 minutes - fidelity = 0.7244; 500 iterations 0.05 - plateau at fidelity = 0.74
# consider subdividing pulse more: +N
iter = 500

for i in range(iter):
    grad_R,grad_phi,grad_omega = num_grad(R,phi,omega_rf,dts) #,grad_dt
    
    R -= learn_rate * grad_R
    phi -= learn_rate * grad_phi
    omega_rf -= learn_rate * grad_omega
    #dts -= learn_rate * grad_dt
    
    R = np.clip(R,0,1)
    phi = phi % (2*np.pi)
    
    if i % 10 == 0 or i == iter-1:
        print(f"Iteration {i}, Fidelity={fidelity(R,phi,omega_rf,dts):.6f}")

Iteration 0, Fidelity=0.352359
Iteration 10, Fidelity=0.385224
Iteration 20, Fidelity=0.412809
Iteration 30, Fidelity=0.435352
Iteration 40, Fidelity=0.453416
Iteration 50, Fidelity=0.467692
Iteration 60, Fidelity=0.478872
Iteration 70, Fidelity=0.487583
Iteration 80, Fidelity=0.494357
Iteration 90, Fidelity=0.499630
Iteration 100, Fidelity=0.503748
Iteration 110, Fidelity=0.506982
Iteration 120, Fidelity=0.509543
Iteration 130, Fidelity=0.511589
Iteration 140, Fidelity=0.513244
Iteration 150, Fidelity=0.514599
Iteration 160, Fidelity=0.515724
Iteration 170, Fidelity=0.516671
Iteration 180, Fidelity=0.517480
Iteration 190, Fidelity=0.518180
Iteration 200, Fidelity=0.518795
Iteration 210, Fidelity=0.519341
Iteration 220, Fidelity=0.519831
Iteration 230, Fidelity=0.520274
Iteration 240, Fidelity=0.520679
Iteration 250, Fidelity=0.521051
Iteration 260, Fidelity=0.521395
Iteration 270, Fidelity=0.521714
Iteration 280, Fidelity=0.522012
Iteration 290, Fidelity=0.522290
Iteration 300, Fideli

In [ ]:
# Gradient Ascent - GRAPE --- INCOMPLETE: learning loop

# a bit different from the basic SMP, the parameter phi is omitted, collapsed into the amplitude which now differs for each orthogonal direction x, y


# constants
n = 2
d = 2**n
w0 = 6250
w1_max = 6250
J = 696
T = 200e-6
N = 50  # maybe add more divisions?
dt = T/N

# fixed hamiltonians
H_J = 2*np.pi*J*(np.kron(Z,Z)/4)
H_x = 2*np.pi*w1_max*(Ix1+Ix2)
H_y = 2*np.pi*w1_max*(Iy1+Iy2)

# initialize
R_x = np.ones(N)
R_y = np.zeros(N)

def p_calc(R_x,R_y):
    P_list = []
    P = U_target.copy()

    for k in reversed(range(N)):
        H = H_J + R_x[k]*H_x + R_y[k]*H_y
        Uk = scipy.linalg.expm(-1j * H * dt)
        P = Uk.conj().T @ P
        P_list.insert(0, P)

    return P_list

def x_calc(R_x,R_y):
    X_list = []
    U = np.eye(d, dtype=complex)

    for k in range(N):
        H = H_J + R_x[k]*H_x + R_y[k]*H_y
        Uk = scipy.linalg.expm(-1j * H * dt)
        U = Uk @ U
        X_list.append(U)

    return X_list, U

def fidelity(U):
    return np.abs(np.trace(U_target.conj().T @ U))/(2**n)

def grape_grad(R_x, R_y):
    
    X_list, U_final = x_calc(R_x, R_y)
    P_list = p_calc(R_x, R_y)
    
    F0 = fidelity(U_final)
    
    for k in range(N):
        Xk = X_list[k]
        Pk = P_list[k]
        
        dX = -1j*dt*H_x @ Xk
        dY = -1j*dt*H_y @ Xk
        
        grad_x = -2*np.real((Pk.conj().T @ dX)*(Xk.conj().T @ Pk))
        grad_y = -2*np.real((Pk.conj().T @ dY)*(Xk.conj().T @ Pk))
        
    return F0, grad_x, grad_y
    
    